In [1]:
import pandas as pd

In [2]:
!pip install nba_api

In [5]:
from nba_api.stats.endpoints import leaguegamefinder

In [37]:
# Get all games from 2025-26 season
gamefinder = leaguegamefinder.LeagueGameFinder(season_nullable='2025-26')
games = gamefinder.get_data_frames()[0]

# Keep only regular season games (GAME_ID starts with 002)
regular_games = games[games["GAME_ID"].str.startswith("002")]

regular_games.head()

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
16,22025,1610612760,OKC,Oklahoma City Thunder,0022500789,2026-02-12,OKC vs. MIL,L,239,93,...,0.800,14,33,47,22,3,6,16,8,-17.0
17,22025,1610612747,LAL,Los Angeles Lakers,0022500791,2026-02-12,LAL vs. DAL,W,242,124,...,0.857,10,34,44,35,8,3,12,20,20.0
18,22025,1610612742,DAL,Dallas Mavericks,0022500791,2026-02-12,DAL @ LAL,L,239,104,...,0.760,4,24,28,22,7,1,15,18,-20.0
19,22025,1610612757,POR,Portland Trail Blazers,0022500790,2026-02-12,POR @ UTA,W,240,135,...,0.758,19,33,52,26,7,7,19,23,16.0
20,22025,1610612749,MIL,Milwaukee Bucks,0022500789,2026-02-12,MIL @ OKC,W,240,110,...,0.714,15,34,49,25,9,6,10,17,17.0


In [39]:
# Get one example game ID
game_id = regular_games.iloc[0]["GAME_ID"]
game_id

'0022500789'

In [41]:
import requests

game_id = regular_games.iloc[0]["GAME_ID"]

url = f"https://cdn.nba.com/static/json/liveData/playbyplay/playbyplay_{game_id}.json"

response = requests.get(url)
data = response.json()

# Extract play-by-play actions
actions = data["game"]["actions"]

# Convert to DataFrame
pbp_df = pd.DataFrame(actions)

pbp_df.head()

,actionNumber,clock,timeActual,period,periodType,actionType,subType,qualifiers,personId,x,...,foulDrawnPersonId,turnoverTotal,pointsTotal,assistPlayerNameInitial,assistPersonId,assistTotal,blockPlayerName,blockPersonId,stealPlayerName,stealPersonId
0,2,PT12M00.00S,2026-02-13T00:43:05.2Z,1,REGULAR,period,start,[],0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4,PT11M56.00S,2026-02-13T00:43:07.5Z,1,REGULAR,jumpball,recovered,[],1641717,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,7,PT11M44.00S,2026-02-13T00:43:20.8Z,1,REGULAR,2pt,Jump Shot,[],1631096,5.732589,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,8,PT11M43.00S,2026-02-13T00:43:21.8Z,1,REGULAR,rebound,defensive,[],1629645,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,9,PT11M23.00S,2026-02-13T00:43:44.6Z,1,REGULAR,foul,offensive,[],1630579,NaN,...,1641717.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [51]:
import re

def clock_to_seconds(clock_str):
    # Extract minutes and seconds using regex
    match = re.match(r"PT(\d+)M(\d+)\.(\d+)S", clock_str)
    if match:
        minutes = int(match.group(1))
        seconds = int(match.group(2))
        return minutes * 60 + seconds
    return None

pbp_df["seconds_remaining_in_period"] = pbp_df["clock"].apply(clock_to_seconds)

pbp_df[["clock", "seconds_remaining_in_period"]].head()

,clock,seconds_remaining_in_period
0,PT12M00.00S,720
1,PT11M56.00S,716
2,PT11M44.00S,704
3,PT11M43.00S,703
4,PT11M23.00S,683


In [64]:
pbp_df["game_seconds_elapsed"] = (
    (pbp_df["period"] - 1) * 720
    + (720 - pbp_df["seconds_remaining_in_period"])
)

pbp_df[["period", "clock", "seconds_remaining_in_period", "game_seconds_elapsed"]].head(10)

,period,clock,seconds_remaining_in_period,game_seconds_elapsed
0,1,PT12M00.00S,720,0
1,1,PT11M56.00S,716,4
2,1,PT11M44.00S,704,16
3,1,PT11M43.00S,703,17
4,1,PT11M23.00S,683,37
5,1,PT11M23.00S,683,37
6,1,PT11M05.00S,665,55
7,1,PT10M46.00S,646,74
8,1,PT10M36.00S,636,84
9,1,PT10M34.00S,634,86


In [68]:
pbp_df["is_def_foul"] = (pbp_df["actionType"] == "foul") & (pbp_df["subType"] != "offensive")

pbp_df[["game_seconds_elapsed", "actionType", "subType", "is_def_foul"]].head(20)

,game_seconds_elapsed,actionType,subType,is_def_foul
0,0,period,start,False
1,4,jumpball,recovered,False
2,16,2pt,Jump Shot,False
3,17,rebound,defensive,False
4,37,foul,offensive,False
5,37,turnover,offensive foul,False
6,55,3pt,Jump Shot,False
7,74,3pt,Jump Shot,False
8,84,3pt,Jump Shot,False
9,86,rebound,defensive,False


In [69]:
pbp_df["is_def_foul"].sum()

np.int64(19)